In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import glob
import os
import fiona
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject
from rasterio.enums import Resampling
import numpy as np
import geopandas as gpd
from scipy.ndimage import sobel
import joblib
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, accuracy_score




src_north = rasterio.open("bng_north.tif")
src_south = rasterio.open("bng_south.tif")

nodata_val = src_north.nodata

elevation_matrix, elevation_transform = merge([src_north, src_south])

target_crs = src_north.crs.to_string()
target_height = elevation_matrix.shape[1]
target_width = elevation_matrix.shape[2]

src_north.close()
src_south.close()


xmin = elevation_transform[2]
x_pixel_size = elevation_transform[0]
xmax = xmin + (target_width * x_pixel_size)
ymax = elevation_transform[5]
y_pixel_size = elevation_transform[4]
ymin = ymax + (target_height * y_pixel_size)

elevation_grid = elevation_matrix[0].astype(np.float32)
if nodata_val is not None:
    elevation_grid[elevation_grid == nodata_val] = np.nan

slope_x = sobel(elevation_grid, axis=1)
slope_y = sobel(elevation_grid, axis=0)
slope = np.hypot(slope_x, slope_y)

inputdir = "chirps_precipitation_2025"
filepaths = sorted(glob.glob(os.path.join(inputdir, "chirps-v2.0.*.tif")))

max_rain = np.zeros((target_height, target_width), dtype=np.float32)
total_rain = np.zeros((target_height, target_width), dtype=np.float32)

for path in filepaths:
    with rasterio.open(path) as src_rain:
        day_grid = np.empty((target_height, target_width), dtype=np.float32)


        reproject(
            source=rasterio.band(src_rain, 1),
            destination=day_grid,
            src_transform=src_rain.transform,
            src_crs=src_rain.crs,
            dst_transform=elevation_transform,
            dst_crs=target_crs,
            resampling=Resampling.bilinear
        )


        max_rain = np.maximum(max_rain, day_grid)
        total_rain += day_grid


mean_rain = total_rain / len(filepaths) if filepaths else np.zeros_like(total_rain)

X_full_grid = pd.DataFrame({
    'elevation': elevation_grid.flatten(),
    'max_rainfall': max_rain.flatten(),
    'mean_rainfall': mean_rain.flatten(),
    'total_rainfall': total_rain.flatten(),
    'slope': slope.flatten(),
})

fiona.drvsupport.supported_drivers['KML'] = 'rw'

gdf_floods_bbmp = gpd.read_file("bbmp_official.kml", driver='KML')

gdf_floods_bbmp = gdf_floods_bbmp.to_crs(target_crs)

y_full_grid = np.zeros(elevation_grid.size, dtype=np.uint8)
flood_indices = []

for geom in gdf_floods_bbmp.geometry:
    if geom.geom_type == "Point":
        lon, lat = geom.x, geom.y


        col, row = ~elevation_transform * (lon, lat)
        row, col = int(np.round(row)), int(np.round(col))

        if 0 <= row < target_height and 0 <= col < target_width:
            flat_index = (row * target_width) + col
            flood_indices.append(flat_index)


if not flood_indices:
    raise ValueError("No valid flood points found in the KML file. Check your data.")

y_full_grid[flood_indices] = 1

flood_indices = np.where(y_full_grid == 1)[0]
non_flood_indices = np.where(y_full_grid == 0)[0]

np.random.seed(50)

sampled_non_flood_indices = np.random.choice(
    non_flood_indices,
    size=len(flood_indices) * 3,
    replace=False
)

final_indices = np.concatenate([flood_indices, sampled_non_flood_indices])

X_balanced = X_full_grid.iloc[final_indices]
y_balanced = y_full_grid[final_indices]


rows_idx = final_indices // target_width
cols_idx = final_indices % target_width
lons_pts = xmin + cols_idx * x_pixel_size
lats_pts = ymax + rows_idx * y_pixel_size

def assign_zone(lat, lon):
    if lat > 13.0 and lon < 77.55:
        return 0  # North-West
    elif lat > 13.0 and lon >= 77.55:
        return 1  # North-East
    elif 12.85 < lat <= 13.0:
        return 2  # Central (merge east+west)
    else:
        return 3  # South

groups = np.array([assign_zone(lat, lon)
                   for lat, lon in zip(lats_pts, lons_pts)])

print("Points per zone:", np.bincount(groups))


logo = LeaveOneGroupOut()
all_preds = np.zeros(len(y_balanced), dtype=np.uint8)
all_true  = np.zeros(len(y_balanced), dtype=np.uint8)
fold_scores = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_balanced, y_balanced, groups)):
    X_tr = X_balanced.iloc[train_idx]
    X_te = X_balanced.iloc[test_idx]
    y_tr = y_balanced[train_idx]
    y_te = y_balanced[test_idx]

    rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                                min_samples_leaf=5, class_weight='balanced',
                                random_state=50, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)

    score = accuracy_score(y_te, y_pred)
    fold_scores.append(score)
    all_preds[test_idx] = y_pred
    all_true[test_idx]  = y_te
    print(f"Zone {fold} held out — Accuracy: {score*100:.2f}%")

print(f"\n================ SPATIAL CV RESULTS ================")
print(f"Mean accuracy: {np.mean(fold_scores)*100:.2f}%")
print(f"Std deviation: {np.std(fold_scores)*100:.2f}%")
print(f"\nAggregated classification report:")
print(classification_report(all_true, all_preds))

# Train final model on ALL data and save
rf_model = RandomForestClassifier(
    n_estimators=200,  # back to 200
    max_depth=10,      # back to 10
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=50,
    n_jobs=-1
)
rf_model.fit(X_balanced, y_balanced)
joblib.dump(rf_model, 'flood_model1.pkl')
print("Final model saved!")




Points per zone: [122 117 105 164]
Zone 0 held out — Accuracy: 89.34%
Zone 1 held out — Accuracy: 82.91%
Zone 2 held out — Accuracy: 59.05%
Zone 3 held out — Accuracy: 68.90%

================ SPATIAL CV RESULTS ================
Mean accuracy: 75.05%
Std deviation: 11.83%

Aggregated classification report:
              precision    recall  f1-score   support

           0       0.87      0.78      0.82       381
           1       0.50      0.66      0.57       127

    accuracy                           0.75       508
   macro avg       0.69      0.72      0.70       508
weighted avg       0.78      0.75      0.76       508

Final model saved!
